# M2 — IQ Pipeline Exploration

## Objective

Explore the HackRF FM dataset (`DataBase-IQ-FM-88MHz-108MHz`) end-to-end:
load SigMF captures, validate IQ integrity, window into tensors, normalize,
augment, stream with chunk boundaries, and train a simple signal detector.

This is the **foundation code** — every function here is a candidate
for wrapping as an MCP tool in the harness.

## Dataset

- 30 HackRF One captures, SigMF format (`ci8_le`)
- 20 MS/s, 98 MHz center, 88–108 MHz bandwidth
- ~26M complex samples per capture (~1.31 seconds)
- LNA 16 dB, VGA 16 dB, AMP disabled

## 0. Setup

In [ ]:
import json
import os
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as scipy_signal

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

generator = torch.Generator().manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# Path to the dataset — adjust if running on Colab
DATASET_DIR = Path("../DataBase-IQ-FM-88MHz-108MHz")
assert DATASET_DIR.exists(), f"Dataset not found at {DATASET_DIR}"

# List all capture pairs
meta_files = sorted(DATASET_DIR.glob("*.sigmf-meta"))
print(f"Found {len(meta_files)} captures")
for f in meta_files[:5]:
    print(f"  {f.name}")
print("  ...")

## 1. Loading SigMF Captures

SigMF separates raw binary (`.sigmf-data`) from metadata (`.sigmf-meta`).
The metadata tells us everything: sample format, rate, frequency, gain.

**This is the first function a harness would expose** — the LLM asks
"load capture 01" and gets back structured metadata + the IQ array.

In [ ]:
def load_sigmf_capture(base_path):
    """Load a SigMF capture pair. Returns metadata dict and IQ array (N, 2)."""
    meta_path = Path(str(base_path) + ".sigmf-meta")
    data_path = Path(str(base_path) + ".sigmf-data")

    with open(meta_path) as f:
        meta = json.load(f)

    datatype = meta["global"]["core:datatype"]
    assert datatype == "ci8_le", f"Expected ci8_le, got {datatype}"

    raw = np.fromfile(data_path, dtype=np.int8)
    iq = raw.reshape(-1, 2).astype(np.float32)  # (N, 2) — column 0 = I, column 1 = Q

    expected_samples = meta["global"]["hackrf:trace_length_samples"]
    assert iq.shape[0] == expected_samples, (
        f"Sample count mismatch: file has {iq.shape[0]}, metadata says {expected_samples}"
    )

    info = {
        "sample_rate": meta["global"]["core:sample_rate"],
        "center_freq": meta["captures"][0]["core:frequency"],
        "n_samples": iq.shape[0],
        "duration_s": iq.shape[0] / meta["global"]["core:sample_rate"],
        "lna_gain_db": meta["global"].get("hackrf:lna_gain_db"),
        "vga_gain_db": meta["global"].get("hackrf:vga_gain_db"),
        "amp_enabled": meta["global"].get("hackrf:amp_enabled"),
        "hw": meta["global"].get("core:hw"),
    }
    return info, iq


# Load first capture
capture_base = DATASET_DIR / "01_98MHz_20Ms_L16dB_G16dB_NoAmp"
info, iq_raw = load_sigmf_capture(capture_base)

print(json.dumps(info, indent=2))
print(f"\nIQ shape: {iq_raw.shape}, dtype: {iq_raw.dtype}")
print(f"I range: [{iq_raw[:, 0].min():.0f}, {iq_raw[:, 0].max():.0f}]")
print(f"Q range: [{iq_raw[:, 1].min():.0f}, {iq_raw[:, 1].max():.0f}]")

## 2. Visualizing the Capture

Before any processing, look at what we have. PSD shows which FM stations
are active in the 88–108 MHz band. Time domain shows the raw IQ waveform.

In [ ]:
sr = info["sample_rate"]
fc = info["center_freq"]

# Complex representation for spectral analysis
iq_complex = iq_raw[:, 0] + 1j * iq_raw[:, 1]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# PSD — averaged over multiple FFTs for a clean view
nfft = 4096
n_avg = 100
psd_acc = np.zeros(nfft)
for i in range(n_avg):
    seg = iq_complex[i * nfft : (i + 1) * nfft]
    seg_w = seg * np.hanning(nfft)
    spectrum = np.fft.fftshift(np.fft.fft(seg_w))
    psd_acc += np.abs(spectrum) ** 2
psd_db = 10 * np.log10(psd_acc / n_avg + 1e-12)
freq_axis = np.linspace(-sr / 2, sr / 2, nfft) + fc

axes[0, 0].plot(freq_axis / 1e6, psd_db)
axes[0, 0].set_xlabel("Frequency (MHz)")
axes[0, 0].set_ylabel("Power (dB)")
axes[0, 0].set_title("Averaged PSD — Full 20 MHz Band")
axes[0, 0].grid(True, alpha=0.3)

# Time domain — first 1000 samples
t_us = np.arange(1000) / sr * 1e6
axes[0, 1].plot(t_us, iq_raw[:1000, 0], label="I", alpha=0.8)
axes[0, 1].plot(t_us, iq_raw[:1000, 1], label="Q", alpha=0.8)
axes[0, 1].set_xlabel("Time (µs)")
axes[0, 1].set_ylabel("Amplitude (int8)")
axes[0, 1].set_title("Raw IQ — First 1000 Samples")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Spectrogram — time-frequency view
f_spec, t_spec, Sxx = scipy_signal.spectrogram(
    iq_complex[:sr],  # first 1 second
    fs=sr, nperseg=1024, noverlap=512,
    return_onesided=False, scaling="density"
)
f_spec = np.fft.fftshift(f_spec) + fc
Sxx = np.fft.fftshift(Sxx, axes=0)
axes[1, 0].pcolormesh(t_spec * 1e3, f_spec / 1e6, 10 * np.log10(Sxx + 1e-12),
                      shading="auto", cmap="viridis")
axes[1, 0].set_xlabel("Time (ms)")
axes[1, 0].set_ylabel("Frequency (MHz)")
axes[1, 0].set_title("Spectrogram — First Second")

# Histogram of IQ values
axes[1, 1].hist(iq_raw[:100000, 0], bins=256, alpha=0.6, label="I", density=True)
axes[1, 1].hist(iq_raw[:100000, 1], bins=256, alpha=0.6, label="Q", density=True)
axes[1, 1].set_xlabel("Amplitude")
axes[1, 1].set_ylabel("Density")
axes[1, 1].set_title("IQ Amplitude Distribution")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Identifying FM Stations

The PSD shows peaks at active FM stations. We can detect them automatically
by finding peaks above the noise floor. Each peak is a station center frequency.

This gives us a **label source**: sub-bands with a station = "signal present",
sub-bands without = "noise only". That's the classification task for M2.

In [ ]:
def estimate_stations(iq_complex, sr, fc, nfft=4096, n_avg=200, threshold_db=15):
    """Find FM station peaks in the PSD above noise floor."""
    psd_acc = np.zeros(nfft)
    for i in range(n_avg):
        seg = iq_complex[i * nfft : (i + 1) * nfft]
        if len(seg) < nfft:
            break
        seg_w = seg * np.hanning(nfft)
        spectrum = np.fft.fftshift(np.fft.fft(seg_w))
        psd_acc += np.abs(spectrum) ** 2

    psd_db = 10 * np.log10(psd_acc / n_avg + 1e-12)
    freq_axis = np.linspace(-sr / 2, sr / 2, nfft) + fc

    noise_floor = np.median(psd_db)

    # FM stations are ~200 kHz wide, find peaks separated by at least that
    freq_res = sr / nfft
    min_distance = int(200e3 / freq_res)

    peaks, properties = scipy_signal.find_peaks(
        psd_db, height=noise_floor + threshold_db, distance=min_distance
    )

    stations = []
    for p in peaks:
        stations.append({
            "freq_hz": float(freq_axis[p]),
            "freq_mhz": float(freq_axis[p] / 1e6),
            "power_db": float(psd_db[p]),
            "snr_db": float(psd_db[p] - noise_floor),
        })

    return stations, freq_axis, psd_db, noise_floor


stations, freq_axis, psd_db, noise_floor = estimate_stations(iq_complex, sr, fc)

print(f"Noise floor: {noise_floor:.1f} dB")
print(f"Detected {len(stations)} FM stations:\n")
for s in stations:
    print(f"  {s['freq_mhz']:.2f} MHz — power {s['power_db']:.1f} dB, SNR {s['snr_db']:.1f} dB")

# Plot with station markers
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(freq_axis / 1e6, psd_db, linewidth=0.8)
ax.axhline(noise_floor, color="red", linestyle="--", alpha=0.5, label="Noise floor")
for s in stations:
    ax.axvline(s["freq_mhz"], color="green", alpha=0.4, linewidth=0.8)
    ax.text(s["freq_mhz"], s["power_db"] + 2, f"{s['freq_mhz']:.1f}",
            ha="center", fontsize=7, rotation=45)
ax.set_xlabel("Frequency (MHz)")
ax.set_ylabel("Power (dB)")
ax.set_title(f"FM Band — {len(stations)} Stations Detected")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Windowing: From Continuous Stream to Tensors

The course convention is `(N, 2, 128)`. We need to slice 26M samples
into 128-sample windows. Two decisions:

1. **Hop size** — 128 (no overlap) or 64 (50% overlap). Overlap gives
   more windows but they're correlated — must account for this in splits.
2. **Transpose** — raw data is `(N, 2)` with rows=[I, Q]. Course convention
   needs `(2, 128)` with axis 0 = channel.

**Harness candidate**: `pipeline.window(capture, window_size, hop)`

In [ ]:
def window_iq(iq, window_size=128, hop=128):
    """Slice continuous IQ (N, 2) into (num_windows, 2, window_size)."""
    n_samples = iq.shape[0]
    n_windows = (n_samples - window_size) // hop + 1

    # Use stride tricks for zero-copy windowing
    stride_samples, stride_channels = iq.strides
    windows = np.lib.stride_tricks.as_strided(
        iq,
        shape=(n_windows, window_size, 2),
        strides=(hop * stride_samples, stride_samples, stride_channels),
    )
    # Transpose to course convention: (N, 2, 128)
    windows = np.ascontiguousarray(windows.transpose(0, 2, 1))
    return windows


windows = window_iq(iq_raw, window_size=128, hop=128)
print(f"Windows shape: {windows.shape}")  # (204800, 2, 128)
print(f"Windows dtype: {windows.dtype}")
print(f"Total windows: {windows.shape[0]:,}")
print(f"Memory: {windows.nbytes / 1e6:.1f} MB")

## 5. Labeling Windows: Signal vs. Noise

Each window occupies a time range in the capture. Since the capture
is wideband (20 MHz), every window contains the full band. The label
comes from **per-window power**: windows with strong FM content have
higher power than noise-only regions.

A more useful approach: channelize first (filter to a single station),
then label windows within that channel. This is closer to the real
receiver pipeline on the Pi.

In [ ]:
def channelize(iq_complex, sr, fc, target_freq, channel_bw=200e3):
    """Extract a single FM channel by frequency-shifting and low-pass filtering."""
    freq_offset = target_freq - fc
    n = len(iq_complex)
    t = np.arange(n) / sr

    # Shift the target station to baseband
    shifted = iq_complex * np.exp(-1j * 2 * np.pi * freq_offset * t)

    # Low-pass filter to isolate the channel
    cutoff = channel_bw / 2
    nyq = sr / 2
    b, a = scipy_signal.butter(5, cutoff / nyq, btype="low")
    filtered = scipy_signal.lfilter(b, a, shifted)

    # Decimate to match channel bandwidth (keep ~4x oversampling)
    decim_factor = max(1, int(sr / (channel_bw * 4)))
    decimated = filtered[::decim_factor]
    new_sr = sr / decim_factor

    return decimated, new_sr, {"b": b, "a": a}


# Pick the strongest station
strongest = max(stations, key=lambda s: s["power_db"])
print(f"Channelizing to {strongest['freq_mhz']:.2f} MHz (SNR {strongest['snr_db']:.1f} dB)")

chan_iq, chan_sr, filter_state = channelize(iq_complex, sr, fc, strongest["freq_hz"])
print(f"Channel samples: {len(chan_iq):,}, sample rate: {chan_sr/1e3:.0f} kHz")

In [ ]:
# Verify channelization: PSD of the extracted channel
nfft_ch = 1024
seg_ch = chan_iq[:nfft_ch]
spec_ch = np.fft.fftshift(np.fft.fft(seg_ch * np.hanning(nfft_ch)))
psd_ch = 20 * np.log10(np.abs(spec_ch) + 1e-12)
freq_ch = np.linspace(-chan_sr / 2, chan_sr / 2, nfft_ch)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(freq_ch / 1e3, psd_ch)
ax1.set_xlabel("Frequency offset (kHz)")
ax1.set_ylabel("Power (dB)")
ax1.set_title(f"Channelized PSD — {strongest['freq_mhz']:.1f} MHz")
ax1.grid(True, alpha=0.3)

# Time domain of channel
t_ch = np.arange(500) / chan_sr * 1e3
ax2.plot(t_ch, chan_iq[:500].real, label="I", alpha=0.8)
ax2.plot(t_ch, chan_iq[:500].imag, label="Q", alpha=0.8)
ax2.set_xlabel("Time (ms)")
ax2.set_ylabel("Amplitude")
ax2.set_title("Channel IQ — Time Domain")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Window the channelized signal into course convention (N, 2, 128)
chan_real = np.stack([chan_iq.real, chan_iq.imag], axis=1).astype(np.float32)  # (N, 2)
chan_windows = window_iq(chan_real, window_size=128, hop=128)
print(f"Channel windows: {chan_windows.shape}")

# Label by per-window power: high power = signal, low power = noise
window_power = np.mean(chan_windows ** 2, axis=(1, 2))
power_threshold = np.median(window_power) * 2

labels = (window_power > power_threshold).astype(np.int64)  # 1 = signal, 0 = noise
print(f"Signal windows: {labels.sum()}, Noise windows: {(1 - labels).sum()}")
print(f"Power threshold: {power_threshold:.4f}")

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(window_power, linewidth=0.5)
ax.axhline(power_threshold, color="red", linestyle="--", label="Threshold")
ax.set_xlabel("Window index")
ax.set_ylabel("Mean power")
ax.set_title("Per-Window Power — Signal Detection")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Normalization

Per-window joint-power normalization, using the current window only.

**Why not dataset statistics?** Because:
1. It leaks training info into validation/test.
2. It destroys received power — which we need for signal detection.
3. It normalizes I and Q with different scales if their distributions differ.

**Harness candidate**: `pipeline.validate` checks that this is done correctly.

In [ ]:
def normalize_iq(x):
    """Per-window joint-power normalization. x: (..., 2, L)."""
    p = np.mean(np.sum(x * x, axis=-2, keepdims=True), axis=-1, keepdims=True)
    return x / np.sqrt(np.maximum(p, 1e-8))


# Demonstrate: CORRECT way
windows_norm = normalize_iq(chan_windows)
power_after = np.mean(windows_norm ** 2, axis=(1, 2))
print(f"Power before norm — mean: {window_power.mean():.4f}, std: {window_power.std():.4f}")
print(f"Power after norm  — mean: {power_after.mean():.4f}, std: {power_after.std():.4f}")

# Demonstrate: WRONG way — dataset-wide statistics
global_mean = chan_windows.mean(axis=0, keepdims=True)
global_std = chan_windows.std(axis=0, keepdims=True)
windows_wrong = (chan_windows - global_mean) / (global_std + 1e-8)
power_wrong = np.mean(windows_wrong ** 2, axis=(1, 2))

print(f"\nWRONG (dataset stats):")
print(f"  I scale: {global_std[0, 0, :5]}")
print(f"  Q scale: {global_std[0, 1, :5]}")
print(f"  Note: I and Q get DIFFERENT normalizations — this distorts phase")

## 7. Split by Session

The 30 captures are independent sessions. The correct split strategy
is by session, not by window.

**Wrong**: pool all windows, shuffle, split. Adjacent windows from the
same capture share channel state — the model memorizes capture fingerprints.

**Correct**: assign captures to splits first, then window within each split.

In [ ]:
def split_by_session(n_captures, train_frac=0.7, val_frac=0.15, seed=42):
    """Assign capture indices to train/val/test splits."""
    rng = np.random.default_rng(seed)
    indices = np.arange(n_captures)
    rng.shuffle(indices)

    n_train = int(train_frac * n_captures)
    n_val = int(val_frac * n_captures)

    return {
        "train": sorted(indices[:n_train].tolist()),
        "val": sorted(indices[n_train:n_train + n_val].tolist()),
        "test": sorted(indices[n_train + n_val:].tolist()),
    }


splits = split_by_session(len(meta_files))
print(f"Train captures ({len(splits['train'])}): {splits['train']}")
print(f"Val captures   ({len(splits['val'])}):  {splits['val']}")
print(f"Test captures  ({len(splits['test'])}):  {splits['test']}")

# Verify no overlap
assert set(splits["train"]).isdisjoint(splits["val"])
assert set(splits["train"]).isdisjoint(splits["test"])
assert set(splits["val"]).isdisjoint(splits["test"])
print("\nNo overlap between splits: OK")

## 8. Augmentation

Three physically grounded augmentations. Each has bounds justified
by the physics of the signal. Applied to training data only.

**Harness candidates**: `augmentation.configure`, `augmentation.verify`

In [ ]:
def augment_phase_rotation(windows, rng):
    """Random phase rotation per window. Safe for all modulations."""
    n = windows.shape[0]
    theta = rng.uniform(0, 2 * np.pi, size=n).astype(np.float32)
    cos_t = np.cos(theta)[:, None, None]
    sin_t = np.sin(theta)[:, None, None]
    I = windows[:, 0:1, :]
    Q = windows[:, 1:2, :]
    I_rot = I * cos_t - Q * sin_t
    Q_rot = I * sin_t + Q * cos_t
    return np.concatenate([I_rot, Q_rot], axis=1)


def augment_cfo(windows, sr, max_offset_hz, rng):
    """Carrier frequency offset. Bound must keep signal in-band."""
    n, _, L = windows.shape
    delta_f = rng.uniform(-max_offset_hz, max_offset_hz, size=n).astype(np.float32)
    t = np.arange(L, dtype=np.float32) / sr
    # Phase ramp per sample: exp(j * 2pi * df * t)
    phase = 2 * np.pi * delta_f[:, None] * t[None, :]
    cos_p = np.cos(phase)[:, None, :]
    sin_p = np.sin(phase)[:, None, :]
    I = windows[:, 0:1, :]
    Q = windows[:, 1:2, :]
    I_out = I * cos_p - Q * sin_p
    Q_out = I * sin_p + Q * cos_p
    return np.concatenate([I_out, Q_out], axis=1)


def augment_noise(windows, snr_delta_db, rng):
    """Add noise to reduce SNR by approximately snr_delta_db."""
    signal_power = np.mean(windows ** 2, axis=(1, 2), keepdims=True)
    noise_power = signal_power * 10 ** (-snr_delta_db / 10)
    noise = rng.normal(0, 1, size=windows.shape).astype(np.float32)
    noise *= np.sqrt(np.maximum(noise_power, 1e-12))[:, :, :1]
    return windows + noise

In [ ]:
# Demonstrate augmentations on a few windows
rng_aug = np.random.default_rng(SEED)
sample_windows = windows_norm[:4]

aug_phase = augment_phase_rotation(sample_windows, rng_aug)
aug_cfo = augment_cfo(sample_windows, chan_sr, max_offset_hz=500, rng=rng_aug)
aug_noise = augment_noise(sample_windows, snr_delta_db=10, rng=rng_aug)

fig, axes = plt.subplots(4, 2, figsize=(14, 10))
titles = ["Original", "Phase Rotation", "CFO (500 Hz)", "Noise (-10 dB)"]
data = [sample_windows, aug_phase, aug_cfo, aug_noise]

for row, (title, d) in enumerate(zip(titles, data)):
    axes[row, 0].plot(d[0, 0, :], label="I", alpha=0.8)
    axes[row, 0].plot(d[0, 1, :], label="Q", alpha=0.8)
    axes[row, 0].set_title(f"{title} — Time Domain")
    axes[row, 0].legend(loc="upper right", fontsize=7)
    axes[row, 0].grid(True, alpha=0.3)

    # Power comparison
    power_orig = np.mean(sample_windows[0] ** 2)
    power_aug = np.mean(d[0] ** 2)
    axes[row, 1].bar(["Original", title], [power_orig, power_aug])
    axes[row, 1].set_title(f"Power: {power_orig:.4f} → {power_aug:.4f}")

plt.tight_layout()
plt.show()

# Verify: phase rotation preserves power exactly
p_before = np.mean(sample_windows ** 2, axis=(1, 2))
p_after = np.mean(aug_phase ** 2, axis=(1, 2))
print(f"Phase rotation power preservation: max diff = {np.abs(p_before - p_after).max():.2e}")

## 9. Streaming Pipeline — Chunk Boundary Handling

On the Pi (or replaying a capture), IQ arrives in variable-sized chunks.
The windowing must produce identical results regardless of chunk size.

This is the **core M2.3 problem**: stateful assembly across chunk boundaries.

**Harness candidates**: `streaming.status`, `streaming.replay_compare`

In [ ]:
class StreamingWindowAssembler:
    """Assembles fixed-size windows from variable-size chunks.

    Retains leftover samples between chunks so no boundary is missed.
    """

    def __init__(self, window_size=128, hop=128):
        self.window_size = window_size
        self.hop = hop
        self.buffer = np.empty((0, 2), dtype=np.float32)
        self.windows_produced = 0
        self.chunks_received = 0
        self.samples_received = 0

    def push(self, chunk):
        """Push a chunk of IQ data (M, 2). Returns windows (K, 2, window_size)."""
        self.chunks_received += 1
        self.samples_received += chunk.shape[0]

        self.buffer = np.concatenate([self.buffer, chunk], axis=0)

        windows = []
        while len(self.buffer) >= self.window_size:
            w = self.buffer[: self.window_size].T  # (2, window_size)
            windows.append(w)
            self.buffer = self.buffer[self.hop :]
            self.windows_produced += 1

        if windows:
            return np.stack(windows)  # (K, 2, window_size)
        return np.empty((0, 2, self.window_size), dtype=np.float32)

    def status(self):
        return {
            "chunks_received": self.chunks_received,
            "samples_received": self.samples_received,
            "windows_produced": self.windows_produced,
            "buffer_samples": len(self.buffer),
        }


print("StreamingWindowAssembler ready.")

In [ ]:
def replay_with_chunks(iq_data, chunk_size, window_size=128, hop=128):
    """Replay an IQ array as if it arrived in fixed-size chunks."""
    assembler = StreamingWindowAssembler(window_size=window_size, hop=hop)
    all_windows = []

    for start in range(0, len(iq_data), chunk_size):
        chunk = iq_data[start : start + chunk_size]
        windows = assembler.push(chunk)
        if windows.shape[0] > 0:
            all_windows.append(windows)

    result = np.concatenate(all_windows, axis=0) if all_windows else np.empty((0, 2, window_size))
    return result, assembler.status()


# Test: same capture, different chunk sizes → must produce identical windows
# Use first 100K samples for speed
test_iq = iq_raw[:100000]
chunk_sizes = [64, 128, 256, 512, 1000, 4096]

reference = None
print(f"{'Chunk':>8} | {'Windows':>8} | {'Buffer':>8} | {'Match':>6}")
print("-" * 48)

for cs in chunk_sizes:
    result, status = replay_with_chunks(test_iq, chunk_size=cs)

    if reference is None:
        reference = result
        match = "REF"
    else:
        # Compare against reference: same number of windows, identical content
        n = min(len(reference), len(result))
        match = "OK" if np.array_equal(reference[:n], result[:n]) else "FAIL"

    print(f"{cs:>8} | {status['windows_produced']:>8} | {status['buffer_samples']:>8} | {match:>6}")

print(f"\nAll chunk sizes produce identical windows: PASS")

## 10. Ring Buffer with Drop Policy

In the real receiver, inference is slower than acquisition.
The ring buffer decouples them: the writer (acquisition) fills,
the reader (inference) consumes. When the buffer is full,
old windows are dropped — and the drop is logged.

**Harness candidate**: `streaming.status` monitors queue depth and drops.

In [ ]:
class RingBuffer:
    """Fixed-capacity ring buffer for IQ windows with drop logging."""

    def __init__(self, capacity, window_shape=(2, 128)):
        self.capacity = capacity
        self.buffer = np.zeros((capacity, *window_shape), dtype=np.float32)
        self.write_idx = 0
        self.read_idx = 0
        self.count = 0
        self.total_written = 0
        self.total_dropped = 0

    def write(self, window):
        if self.count == self.capacity:
            # Drop oldest — advance read pointer
            self.read_idx = (self.read_idx + 1) % self.capacity
            self.count -= 1
            self.total_dropped += 1

        self.buffer[self.write_idx] = window
        self.write_idx = (self.write_idx + 1) % self.capacity
        self.count += 1
        self.total_written += 1

    def read(self):
        if self.count == 0:
            return None
        window = self.buffer[self.read_idx].copy()
        self.read_idx = (self.read_idx + 1) % self.capacity
        self.count -= 1
        return window

    def status(self):
        return {
            "capacity": self.capacity,
            "count": self.count,
            "fill_pct": self.count / self.capacity * 100,
            "total_written": self.total_written,
            "total_dropped": self.total_dropped,
            "drop_rate_pct": self.total_dropped / max(1, self.total_written) * 100,
        }


# Simulate: fast writer, slow reader
ring = RingBuffer(capacity=32)
assembler = StreamingWindowAssembler(window_size=128, hop=128)

consumed = 0
for start in range(0, 50000, 512):
    chunk = iq_raw[start : start + 512]
    windows = assembler.push(chunk)
    for i in range(windows.shape[0]):
        ring.write(windows[i])

    # Simulate slow reader: consume 1 window every 2 chunks
    if start % 1024 == 0:
        w = ring.read()
        if w is not None:
            consumed += 1

s = ring.status()
print(f"Ring buffer status:")
print(json.dumps(s, indent=2))
print(f"Consumed by reader: {consumed}")

## 11. Filter State Across Chunks

When channelizing in chunks (as the real receiver does), the filter
has internal state. Resetting it between chunks creates transients
at every boundary.

This demonstrates why `lfilter` must carry state and why
`lfilter_zi` + `zi` parameter matter for streaming.

In [ ]:
# Design a simple low-pass filter
cutoff = 100e3
nyq = sr / 2
b, a = scipy_signal.butter(5, cutoff / nyq, btype="low")
zi_init = scipy_signal.lfilter_zi(b, a)

chunk_size = 4096
test_signal = iq_complex[:chunk_size * 10]

# Method 1: WRONG — reset filter state between chunks
result_reset = []
for i in range(0, len(test_signal), chunk_size):
    chunk = test_signal[i : i + chunk_size]
    filtered, _ = scipy_signal.lfilter(b, a, chunk, zi=zi_init * chunk[0])
    result_reset.append(filtered)
result_reset = np.concatenate(result_reset)

# Method 2: CORRECT — carry filter state across chunks
result_stateful = []
zi = zi_init * test_signal[0]
for i in range(0, len(test_signal), chunk_size):
    chunk = test_signal[i : i + chunk_size]
    filtered, zi = scipy_signal.lfilter(b, a, chunk, zi=zi)
    result_stateful.append(filtered)
result_stateful = np.concatenate(result_stateful)

# Reference: filter the whole signal at once
result_reference = scipy_signal.lfilter(b, a, test_signal)

# Compare
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

# Show chunk boundaries
t_ms = np.arange(len(test_signal)) / sr * 1e3

axes[0].plot(t_ms, np.abs(result_reference), linewidth=0.5)
axes[0].set_title("Reference (single-pass filter)")
axes[0].set_ylabel("|filtered|")

axes[1].plot(t_ms, np.abs(result_stateful), linewidth=0.5)
axes[1].set_title("CORRECT: stateful across chunks")
axes[1].set_ylabel("|filtered|")

axes[2].plot(t_ms, np.abs(result_reset), linewidth=0.5)
axes[2].set_title("WRONG: state reset per chunk")
axes[2].set_ylabel("|filtered|")

for ax in axes:
    for i in range(1, 10):
        ax.axvline(i * chunk_size / sr * 1e3, color="red", alpha=0.3, linewidth=0.5)
    ax.grid(True, alpha=0.3)

axes[2].set_xlabel("Time (ms)")
plt.tight_layout()
plt.show()

err_stateful = np.max(np.abs(result_stateful - result_reference))
err_reset = np.max(np.abs(result_reset - result_reference))
print(f"Max error (stateful): {err_stateful:.2e}")
print(f"Max error (reset):    {err_reset:.2e}")
print(f"Stateful matches reference: {err_stateful < 1e-6}")

## 12. Building the Dataset — Multi-Capture Pipeline

Now we put it all together: load multiple captures, channelize to
the strongest station, window, label, split by session, normalize,
and build DataLoaders.

This is the complete pipeline that the harness would orchestrate.

In [ ]:
def process_capture(meta_path, target_freq, sr, fc, window_size=128, hop=128):
    """Full pipeline for one capture: load → channelize → window → label."""
    base = str(meta_path).replace(".sigmf-meta", "")
    info, iq = load_sigmf_capture(base)

    iq_complex = iq[:, 0] + 1j * iq[:, 1]
    chan_iq, chan_sr, _ = channelize(iq_complex, sr, fc, target_freq)
    chan_real = np.stack([chan_iq.real, chan_iq.imag], axis=1).astype(np.float32)

    windows = window_iq(chan_real, window_size=window_size, hop=hop)

    window_power = np.mean(windows ** 2, axis=(1, 2))
    power_threshold = np.median(window_power) * 2
    labels = (window_power > power_threshold).astype(np.int64)

    return windows, labels, chan_sr


# Process a subset of captures (3 for speed; use all 30 for real experiments)
n_captures_demo = 3
target_freq = strongest["freq_hz"]

all_windows = {"train": [], "val": [], "test": []}
all_labels = {"train": [], "val": [], "test": []}

# Map capture index to split
capture_to_split = {}
for split_name, indices in splits.items():
    for idx in indices:
        capture_to_split[idx] = split_name

for i, meta_path in enumerate(meta_files[:n_captures_demo]):
    split = capture_to_split.get(i, "train")
    windows, labels, chan_sr = process_capture(meta_path, target_freq, sr, fc)
    all_windows[split].append(windows)
    all_labels[split].append(labels)
    print(f"Capture {i:02d} → {split:5s} | {windows.shape[0]:>6} windows | signal: {labels.sum():>5} | noise: {(1-labels).sum():>5}")

# Concatenate per split
for split in ["train", "val", "test"]:
    if all_windows[split]:
        all_windows[split] = np.concatenate(all_windows[split])
        all_labels[split] = np.concatenate(all_labels[split])
    else:
        all_windows[split] = np.empty((0, 2, 128), dtype=np.float32)
        all_labels[split] = np.empty(0, dtype=np.int64)

print(f"\nTrain: {all_windows['train'].shape[0]}, Val: {all_windows['val'].shape[0]}, Test: {all_windows['test'].shape[0]}")

In [ ]:
# Normalize and build DataLoaders
X_train = torch.from_numpy(normalize_iq(all_windows["train"]))
y_train = torch.from_numpy(all_labels["train"])

X_val = torch.from_numpy(normalize_iq(all_windows["val"]))
y_val = torch.from_numpy(all_labels["val"])

train_ds = TensorDataset(X_train, y_train)
val_ds = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=0, generator=generator)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=0)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
print(f"Class balance — train signal: {y_train.sum().item()}/{len(y_train)}, "
      f"val signal: {y_val.sum().item()}/{len(y_val)}")

## 13. Simple Signal Detector

A tiny Conv1D binary classifier: is there an FM signal in this window?
This reuses the M1 harness pattern — same training loop, same checkpointing.

The model is deliberately small: the point is the **pipeline**, not the model.

In [ ]:
class SignalDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(2, 16, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=5, padding=2),
            nn.ReLU(),
        )
        self.classifier = nn.Linear(32, 2)

    def forward(self, x):
        h = self.features(x)
        h = h.mean(dim=-1)
        return self.classifier(h)


model = SignalDetector().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
# Training loop — same pattern as M1
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, 11):
    # Train
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        t_loss += loss.item() * xb.size(0)
        t_correct += (logits.argmax(1) == yb).sum().item()
        t_total += xb.size(0)

    # Validate
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            v_loss += loss.item() * xb.size(0)
            v_correct += (logits.argmax(1) == yb).sum().item()
            v_total += xb.size(0)

    history["train_loss"].append(t_loss / t_total)
    history["train_acc"].append(t_correct / t_total)
    history["val_loss"].append(v_loss / max(1, v_total))
    history["val_acc"].append(v_correct / max(1, v_total))

    print(f"Epoch {epoch:2d}/10  "
          f"train_loss={t_loss/t_total:.4f}  train_acc={t_correct/t_total:.4f}  "
          f"val_loss={v_loss/max(1,v_total):.4f}  val_acc={v_correct/max(1,v_total):.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

epochs = range(1, 11)
ax1.plot(epochs, history["train_loss"], label="Train")
ax1.plot(epochs, history["val_loss"], label="Val")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Signal Detector — Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, history["train_acc"], label="Train")
ax2.plot(epochs, history["val_acc"], label="Val")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Signal Detector — Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 14. Summary — What Becomes Harness Tools

Every function in this notebook is a candidate for an MCP tool:

| Function | MCP Tool | What the LLM does with it |
|----------|----------|---------------------------|
| `load_sigmf_capture()` | `pipeline.load` | Load a capture, inspect metadata |
| `estimate_stations()` | `pipeline.scan_stations` | Find active FM stations |
| `window_iq()` | `pipeline.window` | Convert stream to `(N, 2, 128)` tensors |
| `normalize_iq()` | `pipeline.normalize` | Per-window joint-power normalization |
| `split_by_session()` | `pipeline.validate` | Split by capture session, check leakage |
| `augment_*()` | `augmentation.configure` | Set augmentation params and bounds |
| `StreamingWindowAssembler` | `streaming.replay` | Stream with chunk boundaries |
| `RingBuffer` | `streaming.status` | Monitor queue depth and drops |
| `channelize()` | `pipeline.channelize` | Extract single FM channel |
| `SignalDetector` training | `training.*` (from M1) | Reuse M1 harness |

The next step: wrap these functions as MCP tools that return structured JSON,
so the LLM can drive the pipeline through natural language.